##### ### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID(s):**     1524206


This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **

## 1. Supervised model training


### Import libraries

In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import matplotlib.pyplot as plt


### Reading and pre-processing data

In [28]:

# Reading and pre-processing data
data = pd.read_csv('Assignment1_data/adult_supervised_train.csv', sep=',')
data = data.dropna()
all_features = data[['age','education-num','capital-gain','capital-loss','hours-per-week','workclass','education','marital-status','occupation','relationship','race','sex','native-country']]
display(all_features)

,age,education-num,capital-gain,capital-loss,hours-per-week,workclass,education,marital-status,occupation,relationship,race,sex,native-country
0,63,4,3471,0,45,Private,7th-8th,Married-civ-spouse,Craft-repair,Husband,White,Male,United-States
1,44,14,7688,0,50,State-gov,Masters,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States
2,18,8,0,0,30,Private,12th,Never-married,Other-service,Own-child,White,Male,Italy
3,39,12,0,0,42,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Husband,White,Male,United-States
4,28,6,0,0,40,Private,10th,Never-married,Handlers-cleaners,Other-relative,White,Male,Mexico
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16275,19,7,0,0,56,Private,11th,Never-married,Craft-repair,Not-in-family,White,Male,United-States
16276,27,13,0,1887,60,Private,Bachelors,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,United-States
16277,45,9,0,0,40,State-gov,HS-grad,Separated,Sales,Unmarried,Black,Female,United-States
16278,25,6,0,0,45,Private,10th,Married-civ-spouse,Craft-repair,Husband,White,Male,United-States


### Splitting into training and testing data

In [29]:
X_train, X_test, y_train, y_test = train_test_split(all_features, data[['income']], test_size=0.2, random_state=42)

X_train_con = X_train[['age','education-num','capital-gain','capital-loss','hours-per-week']]
X_train_cat = X_train[['workclass','education','marital-status','occupation','relationship','race','sex','native-country']]

X_test_con = X_test[['age','education-num','capital-gain','capital-loss','hours-per-week']]
X_test_cat = X_test[['workclass','education','marital-status','occupation','relationship','race','sex','native-country']]

categorical_encoder = OrdinalEncoder()
X_train_cat_encoded = categorical_encoder.fit_transform(X_train_cat)
X_test_cat_encoded = categorical_encoder.transform(X_test_cat)

print(y_test)
label_encoder = OrdinalEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

y_test_encoded

      income
11145   >50K
11808  <=50K
10923  <=50K
14446  <=50K
12105  <=50K
...      ...
14770  <=50K
323    <=50K
12248   >50K
6021   <=50K
1829    >50K

[3016 rows x 1 columns]


array([[1.],
       [0.],
       [0.],
       ...,
       [1.],
       [0.],
       [1.]])

### Model Training

In [30]:
gaussian_nb = GaussianNB()
categorical_nb = CategoricalNB()

gaussian_nb.fit(X_train_con, y_train)
categorical_nb.fit(X_train_cat_encoded, y_train)

continuous_prob = gaussian_nb.predict_log_proba(X_test_con)
categorical_prob = categorical_nb.predict_log_proba(X_test_cat_encoded)

mixed_posterior = continuous_prob + categorical_prob
display(mixed_posterior)
highest_posterior = np.argmax(mixed_posterior, axis=1)
print(highest_posterior)

print(accuracy_score(y_test_encoded, highest_posterior))

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


array([[-2.36309728e+00, -3.45199842e+00],
       [-2.18860548e-03, -1.44900609e+01],
       [-9.66954797e-01, -5.17878180e+00],
       ...,
       [-5.55692020e+00, -1.71685179e+00],
       [-6.01150326e-01, -5.44100303e+00],
       [-6.24812299e-01, -5.32758179e+00]])

[0 0 0 ... 1 0 0]
0.8183023872679045


## 2. Supervised model evaluation

### Smoothing

### Test Prediction

In [32]:
adult_test_data = pd.read_csv('Assignment1_data/adult_test.csv', sep=',')
adult_test_data = adult_test_data.dropna()
display(adult_test_data)

test_label = label_encoder.transform(adult_test_data[['income']])

test_continuous_features = adult_test_data[['age','education-num','capital-gain','capital-loss','hours-per-week']]
test_categorical_features = adult_test_data[['workclass','education','marital-status','occupation','relationship','race','sex','native-country']]

test_categorical_features_encoded = categorical_encoder.transform(test_categorical_features)

test_continuous_prob = gaussian_nb.predict_log_proba(test_continuous_features)
test_categorical_prob = categorical_nb.predict_log_proba(test_categorical_features_encoded)

test_mixed_posterior = test_continuous_prob + test_categorical_prob
test_prediction = np.argmax(test_mixed_posterior, axis=1)
print(accuracy_score(test_prediction, test_label))


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,27,Private,187981,HS-grad,9,Never-married,Handlers-cleaners,Own-child,White,Male,0,0,40,United-States,<=50K
1,55,Private,393768,Assoc-voc,11,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,<=50K
2,38,Private,108726,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,>50K
3,31,Private,180551,HS-grad,9,Married-civ-spouse,Adm-clerical,Wife,White,Female,0,0,40,United-States,>50K
4,51,Self-emp-not-inc,176240,Assoc-acdm,12,Married-civ-spouse,Sales,Husband,White,Male,0,0,40,United-States,>50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16276,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
16277,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
16278,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
16279,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


ValueError: Found unknown categories ['Holand-Netherlands'] in column 7 during transform

## 3. Extending the model with semi-supervised training

### Option 3: Expectation-Maximisation (EM)

## 4. Supervised model evaluation